# Optuna Hyperparameter Tuning with MLflow

## Import Packages

In [1]:
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn import set_config
from sklearn.ensemble import RandomForestClassifier

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (
    StandardScaler,
    MinMaxScaler,
    OneHotEncoder
)

from feature_engine.encoding import CountFrequencyEncoder
from feature_engine.outliers.winsorizer import Winsorizer

import mlflow
import mlflow.sklearn

import optuna
from optuna.samplers import TPESampler

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# suppress optuna's per-trial INFO logs for cleaner output
optuna.logging.set_verbosity(optuna.logging.WARNING)

C:\Users\Ansh\AppData\Roaming\Python\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load the Data

In [2]:
df = pd.read_csv("data/titanic.csv")
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


## Data Cleaning

In [3]:
columns_to_drop = ['passengerid', 'name', 'ticket', 'cabin']

def clean_data(df):
    return (
        df
        .rename(columns=str.lower)
        .drop(columns=columns_to_drop)
        .assign(family=lambda df_: df_['sibsp'] + df_['parch'])
        .drop(columns=['sibsp', 'parch'])
    )

final_df = clean_data(df)
final_df.head()

,survived,pclass,sex,age,fare,embarked,family
0,0,3,male,22.0,7.2500,S,1
1,1,1,female,38.0,71.2833,C,1
2,1,3,female,26.0,7.9250,S,0
3,1,1,female,35.0,53.1000,S,1
4,0,3,male,35.0,8.0500,S,0


## Feature Engineering & Train-Test Split

In [4]:
set_config(transform_output='pandas')

X = final_df.drop(columns=['survived'])
y = final_df['survived']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print('Training shape:', X_train.shape)
print('Testing shape :', X_test.shape)

Training shape: (712, 6)
Testing shape : (179, 6)


## Preprocessing Pipeline

In [5]:
age_pipe = Pipeline(steps=[
    ('impute',   SimpleImputer(strategy='median')),
    ('outliers', Winsorizer(capping_method='gaussian', fold=3)),
    ('scale',    StandardScaler())
])

fare_pipe = Pipeline(steps=[
    ('outliers', Winsorizer(capping_method='iqr', fold=1.5)),
    ('scale',    StandardScaler())
])

embarked_pipe = Pipeline(steps=[
    ('impute',        SimpleImputer(strategy='most_frequent')),
    ('count_encode',  CountFrequencyEncoder(encoding_method='count')),
    ('scale',         MinMaxScaler())
])

preprocessor = ColumnTransformer(transformers=[
    ('age',      age_pipe,                                              ['age']),
    ('fare',     fare_pipe,                                             ['fare']),
    ('embarked', embarked_pipe,                                         ['embarked']),
    ('sex',      OneHotEncoder(sparse_output=False, handle_unknown='ignore'), ['sex']),
    ('family',   MinMaxScaler(),                                        ['family'])
], remainder='passthrough', n_jobs=-1)

## MLflow + Optuna Setup

### How it works
- **Optuna** samples hyperparameters intelligently using the **TPE (Tree-structured Parzen Estimator)** algorithm — much smarter than grid search because it focuses on promising regions of the search space.
- Each Optuna **trial** runs inside a **nested MLflow child run** so every combination is tracked individually.
- At the end, the **best trial's model** is logged in a separate parent MLflow run and registered in the Model Registry.

In [6]:
mlflow.set_tracking_uri('http://localhost:8000')
mlflow.set_experiment('Optuna_Tuning')

<Experiment: artifact_location='mlflow-artifacts:/3', creation_time=1781117224378, effective_trace_archival_retention=None, experiment_id='3', last_update_time=1781117224378, lifecycle_stage='active', name='Optuna_Tuning', tags={}, trace_location=None, workspace='default'>

## Define the Objective Function

In [ ]:
import mlflow
import optuna
from optuna.samplers import TPESampler
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score


# ═══════════════════════════════════════════════════════════════════════
# CELL 1 — Objective Function (called 4 times by study.optimize())
# ═══════════════════════════════════════════════════════════════════════

def objective(trial):
    # Optuna suggests hyperparameter values for this trial
    # Each trial gets different values based on what worked before (TPE)
    params = {
        # This line says "you can pick 100 or 200"
# But it does NOT decide which one
        'n_estimators': trial.suggest_int('n_estimators', 100, 200, step=100),  # 100 or 200
        'max_depth':    trial.suggest_int('max_depth', 2, 3),                   # 2 or 3
        'criterion':    trial.suggest_categorical('criterion', ['gini', 'entropy'])  # gini or entropy
    }

    # Build a fresh pipeline for this trial using the suggested params
    # preprocessor handles scaling/encoding, clf is the random forest
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),                                          # feature preprocessing
        ('clf', RandomForestClassifier(**params, n_jobs=-1, random_state=42))   # model with suggested params
    ])

    # Evaluate the pipeline using 2-fold cross validation
    # splits X_train into 2 parts, trains on one, tests on other, then swaps
    cv_scores = cross_val_score(pipeline, X_train, y_train, cv=2, scoring='accuracy')
    mean_accuracy = cv_scores.mean()  # average of both fold scores

    # Log this trial's params and scores to MLflow as a child run
    # nested=True means it sits under the parent 'Optuna_Study' run
    with mlflow.start_run(run_name='RandomForest_1', nested=True):
        mlflow.log_params(params)                        # log n_estimators, max_depth, criterion
        mlflow.log_metrics({
            'cv_accuracy_mean': cv_scores.mean(),        # average accuracy across folds
            'cv_accuracy_std':  cv_scores.std()          # consistency of scores across folds
        })    # WHENEVER YOU USE 'LOG' ALWAYS PASS A DICTONARY.. 

    # Return accuracy to Optuna — it uses this to decide next trial's params
    return mean_accuracy   # this is not optional optuna decides with this value what parameters to take for next time..
    #this mean_accuracy is filled inside TPE Sampler..

## Run the Optuna Study and Train Best Model on Full Training Set & Log to MLflow

In [ ]:
with mlflow.start_run(run_name='Optuna_Study_1') as parent_run:  # open parent run

    # Create the Optuna study
    # direction='maximize' means Optuna looks for highest accuracy
    # TPESampler learns from past trials to make smarter suggestions
    # seed=42 makes the search reproducible
    study = optuna.create_study(
        direction='maximize',
        sampler=TPESampler(seed=42)  # THIS is what decides which number to pick

    )

    # Run 4 trials — Python stays on this line until ALL 4 trials finish
    # Each trial calls objective() once and waits for it to return
    # Trial 0 → Trial 1 → Trial 2 → Trial 3 → optimize() finishes
    study.optimize(objective, n_trials=4, show_progress_bar=True)

#NOW ALL TRIALS ARE COMPLETED NOW WE WILL LOG OUR BEST MODEL..    
with mlflow.start_run(run_name = 'Best_Random_Model_2') as Bestiiii:
    # After all 4 trials, Optuna knows the best params
    # Build the FINAL pipeline using those best params
    # This pipeline will be trained on the FULL training data (not CV splits)
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('clf', RandomForestClassifier(
            **study.best_params,   # unpack best hyperparameters found by Optuna
            n_jobs=-1,             # use all CPU cores
            random_state=42        # reproducibility
        ))
    ])

    # Train the final model on the complete training set
    # Unlike CV inside objective(), this uses ALL of X_train
    pipeline.fit(X_train, y_train)

    # Log best params to the parent MLflow run
    # dict comprehension adds 'best_' prefix → best_n_estimators, best_max_depth etc.
    mlflow.log_params({f'best_{k}': v for k, v in study.best_params.items()})

    # Log the best CV accuracy achieved across all 4 trials
    mlflow.log_metrics({'best_cv_accuracy': study.best_value})

    # Save the entire trained pipeline (preprocessor + model) to MLflow
    # Can be loaded later for predictions using mlflow.sklearn.load_model()
    mlflow.sklearn.log_model(sk_model=pipeline, artifact_path='RandomForest')


    # create mlflow data
    data_train = mlflow.data.from_pandas(df=X_train,name='training')
    data_train_1 = mlflow.data.from_pandas(df = y_train, name = 'testing')
    # Print summary to console
    print(f"Best trial    : #{study.best_trial.number}")
    print(f"Best accuracy : {study.best_value:.4f}")
    print(f"Best params   : {study.best_params}")

# parent run closes here automatically ✅

Best trial: 0. Best value: 0.81882:  25%|██▌       | 1/4 [00:59<02:58, 59.61s/it]

🏃 View run RandomForest_1 at: http://localhost:8000/#/experiments/3/runs/19f47d4f41654c6b8ce0e544d7bd44e8
🧪 View experiment at: http://localhost:8000/#/experiments/3


Best trial: 0. Best value: 0.81882:  50%|█████     | 2/4 [01:02<00:52, 26.30s/it]

🏃 View run RandomForest_1 at: http://localhost:8000/#/experiments/3/runs/773e0b90e8dc45fdb41e325dc9ce8d64
🧪 View experiment at: http://localhost:8000/#/experiments/3


Best trial: 0. Best value: 0.81882:  75%|███████▌  | 3/4 [01:06<00:16, 16.29s/it]

...

Best trial: 0. Best value: 0.81882: 100%|██████████| 4/4 [01:10<00:00, 17.67s/it]

🏃 View run RandomForest_1 at: http://localhost:8000/#/experiments/3/runs/e449a9be9dce4262862687c4c144121f
🧪 View experiment at: http://localhost:8000/#/experiments/3
🏃 View run Optuna_Study_1 at: http://localhost:8000/#/experiments/3/runs/7d680f06953f41449f976257b6f19230
🧪 View experiment at: http://localhost:8000/#/experiments/3



2026/06/11 15:43:50 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/06/11 15:43:50 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run Best_Random_Model_2 at: http://localhost:8000/#/experiments/3/runs/f3e0d18d3e9842d5bb1f7bde9aced469
🧪 View experiment at: http://localhost:8000/#/experiments/3


AttributeError: 'Series' object has no attribute 'columns'